# Barter Python: Instruments & Market Data

This notebook covers the data layer of the Barter Python bindings:
instruments, indexed lookups, single-exchange streaming, multi-exchange
composition, L1 order books, and the WebSocket integration surface.

## Topics Covered
- Defining spot and perpetual instruments
- `IndexedInstruments` for O(1) indexed lookups
- JSON serialisation of instruments
- Live public-trade streaming via `build_market_stream()`
- L1 order book streaming (best bid/ask)
- Multi-exchange unified streams (12 exchanges supported)
- WebSocket integration surface and current Python boundary

## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [1]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


Using barter from /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python/python


In [ ]:
import json
from collections import Counter

from barter import (
    ExchangeId,
    IndexedInstruments,
    Instrument,
    Subscription,
    build_market_stream,
)


---
## Part 1: Instruments & Index Management


## 1. Define Instruments

Create spot and perpetual instruments from Python while still relying on the same Rust-backed indexed model.

In [3]:
btc_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "btc_usdt", "BTCUSDT", "btc", "usdt")
eth_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "eth_usdt", "ETHUSDT", "eth", "usdt")
btc_perp = Instrument.new_instrument(
    ExchangeId.BINANCE_FUTURES_USD,
    "btc_usdt_perp",
    "BTCUSDT",
    "btc",
    "usdt",
    kind="perpetual",
)

for instrument in [btc_spot, eth_spot, btc_perp]:
    print(instrument)

binance_spot:BTCUSDT (spot)
binance_spot:ETHUSDT (spot)
binance_futures_usd:BTCUSDT (perpetual)


## 2. Build Indexed Lookups

`IndexedInstruments` materializes compact integer indices for exchanges, assets, and instruments.

In [4]:
indexed = IndexedInstruments([btc_spot, eth_spot, btc_perp])

print(indexed)
print("num_exchanges", indexed.num_exchanges)
print("num_assets", indexed.num_assets)
print("exchanges", indexed.exchanges())
print("instruments", indexed.instruments())
print("binance_spot index", indexed.find_exchange_index(ExchangeId.BINANCE_SPOT))

IndexedInstruments(exchanges=2, assets=5, instruments=3)
num_exchanges 2
num_assets 5
exchanges [(0, 'binance_futures_usd'), (1, 'binance_spot')]
instruments [(0, 'btc_usdt_perp', 'BTCUSDT', 'binance_futures_usd', 'perpetual'), (1, 'btc_usdt', 'BTCUSDT', 'binance_spot', 'spot'), (2, 'eth_usdt', 'ETHUSDT', 'binance_spot', 'spot')]
binance_spot index 1


## 3. JSON Serialization

The instrument objects round-trip through JSON cleanly, which is useful when feeding `SystemConfig` or notebook fixtures.

In [5]:
payload = btc_perp.to_json()
round_trip = Instrument.from_json(payload)

print(payload)
print(round_trip)
print(json.loads(payload))

{
  "exchange": "binance_futures_usd",
  "name_internal": "btc_usdt_perp",
  "name_exchange": "BTCUSDT",
  "underlying": {
    "base": {
      "name_internal": "btc",
      "name_exchange": "btc"
    },
    "quote": {
      "name_internal": "usdt",
      "name_exchange": "usdt"
    }
  },
  "quote": "UnderlyingQuote",
  "kind": {
    "perpetual": {
      "contract_size": "1",
      "settlement_asset": {
        "name_internal": "usdt",
        "name_exchange": "usdt"
      }
    }
  },
  "spec": null
}
binance_futures_usd:BTCUSDT (perpetual)
{'exchange': 'binance_futures_usd', 'name_internal': 'btc_usdt_perp', 'name_exchange': 'BTCUSDT', 'underlying': {'base': {'name_internal': 'btc', 'name_exchange': 'btc'}, 'quote': {'name_internal': 'usdt', 'name_exchange': 'usdt'}}, 'quote': 'UnderlyingQuote', 'kind': {'perpetual': {'contract_size': '1', 'settlement_asset': {'name_internal': 'usdt', 'name_exchange': 'usdt'}}}, 'spec': None}


---
## Part 2: Market Data Streaming


The Python binding exposes live streaming via `Subscription` and
`build_market_stream()`:
- **Trades**: `data_kind="trades"` (default) — all 12 exchanges
- **L1 Order Books**: `data_kind="order_book_l1"` — Binance, Kraken, Bybit

## 1. Stream Public Trades

Use top-level `await` in IPython to open the stream and collect a few trade events.

In [6]:
def event_to_dict(event):
    trade = event.trade
    return {
        "time_exchange": event.time_exchange,
        "exchange": str(event.exchange),
        "instrument": event.instrument,
        "kind": event.kind,
        "trade_id": None if trade is None else trade.id,
        "price": None if trade is None else trade.price,
        "amount": None if trade is None else trade.amount,
        "side": None if trade is None else str(trade.side),
    }


async def collect_events(subscriptions, limit=5):
    stream = build_market_stream(subscriptions)
    events = []
    async for event in stream:
        events.append(event_to_dict(event))
        if len(events) >= limit:
            break
    return events


subscriptions = [Subscription("binance_spot", "btc", "usdt")]
trade_events = await collect_events(subscriptions, limit=5)
trade_events

[{'time_exchange': '2026-04-10T23:11:05.420+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6206735494',
  'price': 72814.06,
  'amount': 0.02128,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T23:11:05.590+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6206735495',
  'price': 72814.07,
  'amount': 0.00367,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:05.590+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6206735496',
  'price': 72814.07,
  'amount': 0.00015,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:05.590+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6206735497',
  'price': 72814.07,
  'amount': 8e-05,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:05.590+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot'

## 2. Multiple Instruments on One Stream

The Python binding normalizes all returned items into the same `MarketEvent` shape.

In [7]:
subscriptions = [
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("binance_spot", "eth", "usdt"),
]

more_events = await collect_events(subscriptions, limit=8)
more_events

[{'time_exchange': '2026-04-10T23:11:07.180+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3922472719',
  'price': 2239.71,
  'amount': 0.003,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:07.180+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3922472720',
  'price': 2239.71,
  'amount': 0.0023,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:07.180+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3922472721',
  'price': 2239.71,
  'amount': 0.0023,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:07.180+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3922472722',
  'price': 2239.71,
  'amount': 0.0023,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T23:11:07.180+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kin

## 3. Stream L1 Order Books (Best Bid/Ask)

Use `data_kind="order_book_l1"` for top-of-book data. This can be mixed with trades in the same stream.

In [8]:
async def collect_mixed(subscriptions, limit=8):
    stream = build_market_stream(subscriptions)
    rows = []
    async for event in stream:
        row = {
            "exchange": str(event.exchange),
            "instrument": event.instrument,
            "kind": event.kind,
        }
        if event.kind == "trade" and event.trade:
            row["price"] = event.trade.price
            row["amount"] = event.trade.amount
        elif event.kind == "order_book_l1":
            row["best_bid"] = event.best_bid_price
            row["best_ask"] = event.best_ask_price
        rows.append(row)
        if len(rows) >= limit:
            break
    return rows

# Mix trades and L1 order book data from Binance
l1_events = await collect_mixed([
    Subscription("binance_spot", "btc", "usdt", data_kind="trades"),
    Subscription("binance_spot", "btc", "usdt", data_kind="order_book_l1"),
], limit=8)
l1_events

[{'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.06,
  'amount': 0.007},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'order_book_l1',
  'best_bid': 72814.06,
  'best_ask': 72814.07},
 {'exchange': 'binance_spot',
  'instrument

---
## Part 3: Multi-Exchange Streams


## 1. Build a Unified Stream

The Python API accepts multiple subscriptions and yields a unified async stream of `MarketEvent` values.

In [9]:
async def collect_events(subscriptions, limit=12):
    stream = build_market_stream(subscriptions)
    rows = []
    async for event in stream:
        rows.append({
            "exchange": str(event.exchange),
            "instrument": event.instrument,
            "kind": event.kind,
            "price": None if event.trade is None else event.trade.price,
            "amount": None if event.trade is None else event.trade.amount,
        })
        if len(rows) >= limit:
            break
    return rows


subscriptions = [
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("binance_futures_usd", "btc", "usdt", instrument_kind="perpetual"),
    Subscription("coinbase", "eth", "usd"),
]

events = await collect_events(subscriptions)
events

[{'exchange': 'binance_futures_usd',
  'instrument': 'btc_usdt_perpetual',
  'kind': 'trade',
  'price': 72776.1,
  'amount': 0.006},
 {'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'price': 2241.07,
  'amount': 0.003055},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 0.0128},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 7e-05},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 7e-05},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 8e-05},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 8e-05},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 72814.07,
  'amount': 8e-05},
 {'exch

## 2. Aggregate the Joined Stream

Because all items share one shape, plain Python grouping works well once the events are in memory.

In [10]:
by_exchange = Counter(row["exchange"] for row in events)
by_instrument = Counter(row["instrument"] for row in events)

by_exchange, by_instrument

(Counter({'binance_spot': 9, 'binance_futures_usd': 2, 'coinbase': 1}),
 Counter({'btc_usdt_spot': 9, 'btc_usdt_perpetual': 2, 'eth_usd_spot': 1}))

## 3. Python Binding Differences From Rust

The Rust `Streams::builder_multi()` API exposes more typed composition options. The current Python layer keeps the surface smaller: build a list of `Subscription` specs and consume the unified stream directly.

---
## Part 4: WebSocket Integration Surface


## Current Python Boundary

The Rust `barter-integration` crate exposes low-level transformer, validator, and transport traits. Those hooks are not part of the public Python package yet. In Python, the supported integration surface is the normalized async stream returned by `build_market_stream()`.

In [11]:
def snapshot_event(event):
    trade = event.trade
    return {
        "time_exchange": event.time_exchange,
        "time_received": event.time_received,
        "exchange": str(event.exchange),
        "instrument": event.instrument,
        "kind": event.kind,
        "trade": None if trade is None else {
            "id": trade.id,
            "price": trade.price,
            "amount": trade.amount,
            "side": str(trade.side),
        },
    }


async def capture(subscription, limit=3):
    stream = build_market_stream([subscription])
    rows = []
    async for event in stream:
        rows.append(snapshot_event(event))
        if len(rows) >= limit:
            break
    return rows


rows = await capture(Subscription("binance_spot", "btc", "usdt"), limit=3)
rows

[{'time_exchange': '2026-04-10T23:11:12.595+00:00',
  'time_received': '2026-04-10T23:11:11.121426501+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6206735514',
   'price': 72814.07,
   'amount': 0.00078,
   'side': 'buy'}},
 {'time_exchange': '2026-04-10T23:11:12.639+00:00',
  'time_received': '2026-04-10T23:11:11.163454468+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6206735515',
   'price': 72814.07,
   'amount': 0.00827,
   'side': 'buy'}},
 {'time_exchange': '2026-04-10T23:11:13.027+00:00',
  'time_received': '2026-04-10T23:11:11.552682398+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6206735516',
   'price': 72814.06,
   'amount': 0.00015,
   'side': 'sell'}}]

## Event-Normalization Pattern

The practical Python pattern is: open a stream, normalize each `MarketEvent` into plain Python data, and then plug that into your notebook analysis, async app, or downstream pipeline.

In [12]:
async def capture_many(subscriptions, limit=6):
    stream = build_market_stream(subscriptions)
    rows = []
    async for event in stream:
        rows.append(snapshot_event(event))
        if len(rows) >= limit:
            break
    return rows


rows = await capture_many([
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("coinbase", "eth", "usd"),
], limit=6)
rows

[{'time_exchange': '2026-04-10T23:11:13.495492+00:00',
  'time_received': '2026-04-10T23:11:12.395408521+00:00',
  'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'trade': {'id': '796429380',
   'price': 2241.08,
   'amount': 0.021163,
   'side': 'sell'}},
 {'time_exchange': '2026-04-10T23:11:14.118270+00:00',
  'time_received': '2026-04-10T23:11:12.582833582+00:00',
  'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'trade': {'id': '796429381',
   'price': 2241.07,
   'amount': 0.00924633,
   'side': 'buy'}},
 {'time_exchange': '2026-04-10T23:11:15.488815+00:00',
  'time_received': '2026-04-10T23:11:13.957462497+00:00',
  'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'trade': {'id': '796429382',
   'price': 2241.07,
   'amount': 0.011924,
   'side': 'buy'}},
 {'time_exchange': '2026-04-10T23:11:15.804+00:00',
  'time_received': '2026-04-10T23:11:14.396416935+00:00',
  'exchange': 'binance_spot',
  

## Supported Exchanges

| Exchange | Python Name | Trades | L1 Book |
|----------|-------------|--------|---------|
| Binance Spot | `binance_spot` | Yes | Yes |
| Binance Futures | `binance_futures_usd` | Yes | Yes |
| OKX | `okx` | Yes | — |
| Coinbase | `coinbase` | Yes | — |
| Kraken | `kraken` | Yes | Yes |
| Bitfinex | `bitfinex` | Yes | — |
| Bitmex | `bitmex` | Yes | — |
| Bybit Spot | `bybit_spot` | Yes | Yes |
| Bybit Perpetuals | `bybit_perpetuals_usd` | Yes | Yes |
| Gateio Spot | `gateio_spot` | Yes | — |
| Gateio Perpetuals | `gateio_perpetuals_usd` | Yes | — |
| Gateio Futures | `gateio_futures_usd` | Yes | — |

## What Rust Still Has That Python Does Not

- L2 order book depth with `OrderBookMap` background manager
- Custom transformer and validator traits
- Direct access to protocol-level websocket plumbing

Those remain Rust-only for now.